# 07 - DVAE on EMG (disentangled sex / weight latents)

EMG mirror of notebook 06. Two parallel encoders produce `z_agn` (weight-bearing) and `z_sex` (sex-bearing). Decoder reconstructs from `concat(z_agn, z_sex)`. Gradient-reversal adversarial heads drive out sex from `z_agn` and weight from `z_sex`.

Same hyperparameters as 06; only difference is 8-channel input and smaller encoder bases (matching 05 vs 04 scaling).

Outputs to `VAE/Results/dvae_emg/`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    mean_absolute_error, mean_squared_error, confusion_matrix
)

ROOT = Path(r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project")
TENSOR_PATH = ROOT / "VAE" / "Tensors" / "lifts_emg.npz"
OUT_DIR = ROOT / "VAE" / "Results" / "dvae_emg"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Latent sizes and loss weights identical to 06 (IMU DVAE)
LATENT_AGN = 32
LATENT_SEX = 8
BETA = 0.1
LAMBDA_MAIN = 10.0
LAMBDA_ADV = 1.0
GRL_LAMBDA = 1.0
BATCH_SIZE = 64
EPOCHS = 60
LR = 1e-3
WEIGHT_DECAY = 1e-5
LOAD_BINS = np.array([2.3, 4.5, 6.8, 9.1, 11.3])

Device: cpu


In [2]:
# ---- load + drop NaN lifts (EMG should have none, but keep the safeguard) ----
data = np.load(TENSOR_PATH, allow_pickle=True)
X = data["X"].astype(np.float32)
participant = data["participant"]
sex = data["sex"]
box_mass = data["box_mass"].astype(np.float32)
channel_names = data["channel_names"]

nan_lift = np.isnan(X).any(axis=(1, 2))
print(f"Dropping {int(nan_lift.sum())} / {len(X)} lifts with NaN")
if nan_lift.any():
    u, c = np.unique(participant[nan_lift], return_counts=True)
    print("  by subject:", dict(zip(u.tolist(), c.tolist())))
keep = ~nan_lift
X = X[keep]; participant = participant[keep]; sex = sex[keep]; box_mass = box_mass[keep]

def mass_to_class(m): return int(np.argmin(np.abs(LOAD_BINS - m)))
y_cls = np.array([mass_to_class(m) for m in box_mass], dtype=np.int64)

sex_labels = np.array(["Female", "Male"])
sex_to_idx = {s: i for i, s in enumerate(sex_labels)}
y_sex = np.array([sex_to_idx[s] for s in sex], dtype=np.int64)
print("Sex dist:", dict(zip(*np.unique(y_sex, return_counts=True))))
print("X shape:", X.shape, "subjects:", len(np.unique(participant)))

Dropping 0 / 4750 lifts with NaN
Sex dist: {0: 2351, 1: 2399}
X shape: (4750, 400, 8) subjects: 40


In [3]:
# ---- subject-grouped 80/20 split, sex-stratified ----
subjects = np.unique(participant)
subj_sex = {p: sex[participant == p][0] for p in subjects}
rng = np.random.default_rng(SEED)
test_subjects = []
for s_label in np.unique(list(subj_sex.values())):
    subs_s = np.array([p for p, s in subj_sex.items() if s == s_label])
    rng.shuffle(subs_s)
    n_test = max(1, int(round(0.2 * len(subs_s))))
    test_subjects.extend(subs_s[:n_test].tolist())
test_subjects = set(test_subjects)
train_subjects = set(subjects) - test_subjects
train_mask = np.array([p in train_subjects for p in participant])
test_mask = ~train_mask
print(f"Train subjects: {len(train_subjects)} lifts: {train_mask.sum()}  |  Test subjects: {len(test_subjects)} lifts: {test_mask.sum()}")

Train subjects: 32 lifts: 3791  |  Test subjects: 8 lifts: 959


In [4]:
# ---- channel z-score and target normalization (fit on train) ----
X_train_raw = X[train_mask]
ch_mean = X_train_raw.reshape(-1, X.shape[-1]).mean(axis=0)
ch_std = X_train_raw.reshape(-1, X.shape[-1]).std(axis=0) + 1e-6
Xn = (X - ch_mean) / ch_std

y_mean = float(box_mass[train_mask].mean())
y_std = float(box_mass[train_mask].std() + 1e-6)
y_norm = (box_mass - y_mean) / y_std

X_train = torch.from_numpy(Xn[train_mask]).transpose(1, 2)
X_test = torch.from_numpy(Xn[test_mask]).transpose(1, 2)
yw_train = torch.from_numpy(y_norm[train_mask]).float()
yw_test = torch.from_numpy(y_norm[test_mask]).float()
ys_train = torch.from_numpy(y_sex[train_mask])
ys_test = torch.from_numpy(y_sex[test_mask])
cls_train = y_cls[train_mask]

class_counts = np.bincount(cls_train, minlength=len(LOAD_BINS))
sample_w = 1.0 / class_counts[cls_train]
sampler = WeightedRandomSampler(weights=torch.from_numpy(sample_w).double(),
                                num_samples=len(sample_w), replacement=True)
train_loader = DataLoader(TensorDataset(X_train, yw_train, ys_train),
                          batch_size=BATCH_SIZE, sampler=sampler)
test_loader = DataLoader(TensorDataset(X_test, yw_test, ys_test),
                         batch_size=BATCH_SIZE, shuffle=False)
print("X_train:", X_train.shape, "X_test:", X_test.shape)

X_train: torch.Size([3791, 8, 400]) X_test: torch.Size([959, 8, 400])


In [5]:
# ---- gradient reversal layer + model ----
class GRLFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_, None
def grl(x, lambda_=1.0):
    return GRLFn.apply(x, lambda_)

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, k, s, p):
        super().__init__()
        self.net = nn.Sequential(nn.Conv1d(in_c, out_c, k, s, p), nn.BatchNorm1d(out_c), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class DeconvBlock(nn.Module):
    def __init__(self, in_c, out_c, k, s, p, op):
        super().__init__()
        self.net = nn.Sequential(nn.ConvTranspose1d(in_c, out_c, k, s, p, output_padding=op), nn.BatchNorm1d(out_c), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class Encoder(nn.Module):
    def __init__(self, n_channels, seq_len, latent_dim, base=32):
        super().__init__()
        self.enc = nn.Sequential(
            ConvBlock(n_channels, base, 7, 2, 3),
            ConvBlock(base, base*2, 5, 2, 2),
            ConvBlock(base*2, base*4, 3, 2, 1),
        )
        self.out_len = seq_len // 8
        self.flat_dim = base*4 * self.out_len
        self.fc_mu = nn.Linear(self.flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flat_dim, latent_dim)
    def forward(self, x):
        h = self.enc(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

class Decoder(nn.Module):
    def __init__(self, n_channels, seq_len, latent_total, base=32):
        super().__init__()
        self.out_len = seq_len // 8
        self.fc = nn.Linear(latent_total, base*4 * self.out_len)
        self.dec = nn.Sequential(
            DeconvBlock(base*4, base*2, 3, 2, 1, 1),
            DeconvBlock(base*2, base, 5, 2, 2, 1),
            nn.ConvTranspose1d(base, n_channels, 7, 2, 3, output_padding=1),
        )
        self.base = base
    def forward(self, z):
        return self.dec(self.fc(z).view(-1, self.base*4, self.out_len))

class DVAE(nn.Module):
    # EMG bases: base=32 for agn encoder/decoder, base=16 for sex encoder (mirrors 05 vs 04 scaling)
    def __init__(self, n_channels, seq_len=400, latent_agn=32, latent_sex=8, n_load_classes=5):
        super().__init__()
        self.enc_agn = Encoder(n_channels, seq_len, latent_agn, base=32)
        self.enc_sex = Encoder(n_channels, seq_len, latent_sex, base=16)
        self.dec = Decoder(n_channels, seq_len, latent_agn + latent_sex, base=32)
        self.reg_main = nn.Sequential(nn.Linear(latent_agn, 64), nn.ReLU(inplace=True), nn.Linear(64, 1))
        self.cls_main = nn.Sequential(nn.Linear(latent_sex, 32), nn.ReLU(inplace=True), nn.Linear(32, 2))
        self.cls_adv = nn.Sequential(nn.Linear(latent_agn, 32), nn.ReLU(inplace=True), nn.Linear(32, 2))
        self.reg_adv = nn.Sequential(nn.Linear(latent_sex, 32), nn.ReLU(inplace=True), nn.Linear(32, 1))

    def reparam(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)

    def forward(self, x, grl_lambda=1.0):
        mu_a, lv_a = self.enc_agn(x)
        mu_s, lv_s = self.enc_sex(x)
        z_a = self.reparam(mu_a, lv_a)
        z_s = self.reparam(mu_s, lv_s)
        x_rec = self.dec(torch.cat([z_a, z_s], dim=1))
        w_main = self.reg_main(z_a).squeeze(-1)
        s_main = self.cls_main(z_s)
        s_adv = self.cls_adv(grl(z_a, grl_lambda))
        w_adv = self.reg_adv(grl(z_s, grl_lambda)).squeeze(-1)
        return x_rec, (mu_a, lv_a, mu_s, lv_s), (w_main, s_main, s_adv, w_adv)

model = DVAE(n_channels=X.shape[-1], seq_len=X.shape[1],
             latent_agn=LATENT_AGN, latent_sex=LATENT_SEX).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 811,374


In [6]:
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
history = []

def kl(mu, logvar):
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr = {"rec": 0.0, "kl": 0.0, "reg": 0.0, "cls": 0.0, "adv_cls": 0.0, "adv_reg": 0.0,
          "n": 0, "w_abs": 0.0, "s_correct": 0}
    for xb, ywb, ysb in train_loader:
        xb = xb.to(device); ywb = ywb.to(device); ysb = ysb.to(device)
        opt.zero_grad()
        x_rec, (mu_a, lv_a, mu_s, lv_s), (w_main, s_main, s_adv, w_adv) = model(xb, grl_lambda=GRL_LAMBDA)
        L_rec = F.mse_loss(x_rec, xb)
        L_kl = kl(mu_a, lv_a) + kl(mu_s, lv_s)
        L_reg = F.mse_loss(w_main, ywb)
        L_cls = F.cross_entropy(s_main, ysb)
        L_adv_cls = F.cross_entropy(s_adv, ysb)
        L_adv_reg = F.mse_loss(w_adv, ywb)
        L = L_rec + BETA*L_kl + LAMBDA_MAIN*(L_reg + L_cls) + LAMBDA_ADV*(L_adv_cls + L_adv_reg)
        L.backward(); opt.step()
        bs = xb.size(0)
        tr["rec"] += L_rec.item()*bs; tr["kl"] += L_kl.item()*bs
        tr["reg"] += L_reg.item()*bs; tr["cls"] += L_cls.item()*bs
        tr["adv_cls"] += L_adv_cls.item()*bs; tr["adv_reg"] += L_adv_reg.item()*bs
        tr["w_abs"] += (w_main - ywb).abs().sum().item()
        tr["s_correct"] += (s_main.argmax(1) == ysb).sum().item()
        tr["n"] += bs

    model.eval()
    te = {"rec": 0.0, "kl": 0.0, "reg": 0.0, "cls": 0.0, "adv_cls": 0.0, "adv_reg": 0.0,
          "n": 0, "w_abs": 0.0, "s_correct": 0, "s_from_agn_correct": 0}
    with torch.no_grad():
        for xb, ywb, ysb in test_loader:
            xb = xb.to(device); ywb = ywb.to(device); ysb = ysb.to(device)
            x_rec, (mu_a, lv_a, mu_s, lv_s), (w_main, s_main, s_adv, w_adv) = model(xb, grl_lambda=GRL_LAMBDA)
            L_rec = F.mse_loss(x_rec, xb)
            L_kl = kl(mu_a, lv_a) + kl(mu_s, lv_s)
            L_reg = F.mse_loss(w_main, ywb)
            L_cls = F.cross_entropy(s_main, ysb)
            L_adv_cls = F.cross_entropy(s_adv, ysb)
            L_adv_reg = F.mse_loss(w_adv, ywb)
            bs = xb.size(0)
            te["rec"] += L_rec.item()*bs; te["kl"] += L_kl.item()*bs
            te["reg"] += L_reg.item()*bs; te["cls"] += L_cls.item()*bs
            te["adv_cls"] += L_adv_cls.item()*bs; te["adv_reg"] += L_adv_reg.item()*bs
            te["w_abs"] += (w_main - ywb).abs().sum().item()
            te["s_correct"] += (s_main.argmax(1) == ysb).sum().item()
            te["s_from_agn_correct"] += (s_adv.argmax(1) == ysb).sum().item()
            te["n"] += bs

    row = {"epoch": epoch,
           "tr_rec": tr["rec"]/tr["n"], "tr_kl": tr["kl"]/tr["n"],
           "tr_reg": tr["reg"]/tr["n"], "tr_cls": tr["cls"]/tr["n"],
           "tr_advcls": tr["adv_cls"]/tr["n"], "tr_advreg": tr["adv_reg"]/tr["n"],
           "tr_sex_acc": tr["s_correct"]/tr["n"],
           "te_rec": te["rec"]/te["n"], "te_kl": te["kl"]/te["n"],
           "te_reg": te["reg"]/te["n"], "te_cls": te["cls"]/te["n"],
           "te_advcls": te["adv_cls"]/te["n"], "te_advreg": te["adv_reg"]/te["n"],
           "te_sex_main_acc": te["s_correct"]/te["n"],
           "te_sex_from_agn_acc": te["s_from_agn_correct"]/te["n"]}
    history.append(row)
    if epoch % 5 == 0 or epoch == 1:
        print(f"ep{epoch:3d} | rec {row['tr_rec']:.3f} kl {row['tr_kl']:.3f} reg {row['tr_reg']:.3f} cls {row['tr_cls']:.3f} "
              f"adv_c {row['tr_advcls']:.3f} adv_r {row['tr_advreg']:.3f} | "
              f"te_reg {row['te_reg']:.3f} sex_main {row['te_sex_main_acc']:.3f} sex_from_agn {row['te_sex_from_agn_acc']:.3f}")

pd.DataFrame(history).to_csv(OUT_DIR / "history.csv", index=False)

ep  1 | rec 0.925 kl 341.423 reg 0.638 cls 0.667 adv_c 0.714 adv_r 16.096 | te_reg 0.871 sex_main 0.684 sex_from_agn 0.533
ep  5 | rec 0.756 kl 85.229 reg 0.422 cls 0.622 adv_c 0.687 adv_r 7.120 | te_reg 0.829 sex_main 0.711 sex_from_agn 0.509
ep 10 | rec 0.656 kl 14.734 reg 0.249 cls 0.537 adv_c 0.693 adv_r 1.176 | te_reg 0.830 sex_main 0.757 sex_from_agn 0.599
ep 15 | rec 0.611 kl 17.041 reg 0.136 cls 0.460 adv_c 0.682 adv_r 2.495 | te_reg 0.894 sex_main 0.749 sex_from_agn 0.576
ep 20 | rec 0.680 kl 10.581 reg 0.433 cls 0.417 adv_c 0.681 adv_r 1.147 | te_reg 0.833 sex_main 0.762 sex_from_agn 0.560
ep 25 | rec 0.647 kl 11.553 reg 0.519 cls 0.346 adv_c 0.679 adv_r 1.120 | te_reg 0.854 sex_main 0.766 sex_from_agn 0.571
ep 30 | rec 0.645 kl 13.506 reg 0.538 cls 0.369 adv_c 0.674 adv_r 1.532 | te_reg 0.932 sex_main 0.705 sex_from_agn 0.582
ep 35 | rec 0.593 kl 13.613 reg 0.439 cls 0.302 adv_c 0.657 adv_r 1.381 | te_reg 0.816 sex_main 0.632 sex_from_agn 0.626
ep 40 | rec 0.550 kl 10.025 re

In [7]:
# ---- inference: predict weight from z_agn (mu_a) only ----
model.eval()
all_mu_a, all_mu_s, all_pred_norm, all_sex_main, all_sex_adv = [], [], [], [], []
with torch.no_grad():
    for xb, ywb, ysb in test_loader:
        xb = xb.to(device)
        mu_a, _ = model.enc_agn(xb)
        mu_s, _ = model.enc_sex(xb)
        w_pred = model.reg_main(mu_a).squeeze(-1)
        s_main = model.cls_main(mu_s).argmax(1)
        s_adv = model.cls_adv(mu_a).argmax(1)
        all_mu_a.append(mu_a.cpu().numpy())
        all_mu_s.append(mu_s.cpu().numpy())
        all_pred_norm.append(w_pred.cpu().numpy())
        all_sex_main.append(s_main.cpu().numpy())
        all_sex_adv.append(s_adv.cpu().numpy())

mu_a_test = np.concatenate(all_mu_a)
mu_s_test = np.concatenate(all_mu_s)
pred_norm = np.concatenate(all_pred_norm)
sex_main_pred = np.concatenate(all_sex_main)
sex_adv_pred = np.concatenate(all_sex_adv)

mass_pred = pred_norm * y_std + y_mean
mass_true = box_mass[test_mask]
y_true_cls = np.array([mass_to_class(m) for m in mass_true])
y_pred_cls = np.array([mass_to_class(m) for m in mass_pred])

mae = mean_absolute_error(mass_true, mass_pred)
rmse = np.sqrt(mean_squared_error(mass_true, mass_pred))
acc = accuracy_score(y_true_cls, y_pred_cls)
bal_acc = balanced_accuracy_score(y_true_cls, y_pred_cls)
sex_main_acc = accuracy_score(y_sex[test_mask], sex_main_pred)
sex_adv_acc = accuracy_score(y_sex[test_mask], sex_adv_pred)

print(f"Weight  MAE={mae:.3f} kg  RMSE={rmse:.3f} kg  nearest-class acc={acc:.3f}  bal_acc={bal_acc:.3f}")
print(f"Sex classifier (from z_sex): acc={sex_main_acc:.3f}  (should be high)")
print(f"Sex classifier (from z_agn): acc={sex_adv_acc:.3f}  (should be near 0.5 if disentangled)")
print("\nMean predicted kg by true class:")
for k, m in enumerate(LOAD_BINS):
    sel = y_true_cls == k
    if sel.any():
        print(f"  true={m:5.2f} kg  n={int(sel.sum()):3d}  pred_mean={mass_pred[sel].mean():.3f}  pred_std={mass_pred[sel].std():.3f}")
print(f"\npred/true std ratio: {mass_pred.std()/mass_true.std():.3f}")

Weight  MAE=2.371 kg  RMSE=2.985 kg  nearest-class acc=0.309  bal_acc=0.309
Sex classifier (from z_sex): acc=0.706  (should be high)
Sex classifier (from z_agn): acc=0.631  (should be near 0.5 if disentangled)

Mean predicted kg by true class:
  true= 2.30 kg  n=192  pred_mean=4.598  pred_std=2.211
  true= 4.50 kg  n=192  pred_mean=6.287  pred_std=2.457
  true= 6.80 kg  n=191  pred_mean=7.034  pred_std=2.244
  true= 9.10 kg  n=192  pred_mean=7.829  pred_std=2.066
  true=11.30 kg  n=192  pred_mean=8.210  pred_std=2.144

pred/true std ratio: 0.804


In [8]:
# ---- fairness by sex + Rahman-style SP/PRD/NRD + save ----
test_sex = sex[test_mask]
signed_err = mass_pred - mass_true
abs_err = np.abs(signed_err)

fair_rows = []
for s_label in np.unique(test_sex):
    m = test_sex == s_label
    fair_rows.append({
        "sex": s_label, "n": int(m.sum()),
        "mae_kg": float(abs_err[m].mean()),
        "rmse_kg": float(np.sqrt((signed_err[m] ** 2).mean())),
        "mean_signed_err_kg": float(signed_err[m].mean()),
        "over_rate": float((signed_err[m] > 0).mean()),
        "under_rate": float((signed_err[m] < 0).mean()),
        "nearest_acc": accuracy_score(y_true_cls[m], y_pred_cls[m]),
    })
fair_df = pd.DataFrame(fair_rows)
print(fair_df.to_string(index=False))

f_mask = test_sex == "Female"; m_mask = test_sex == "Male"
SP = float(mass_pred[f_mask].mean() - mass_pred[m_mask].mean())
PRD = float(abs(np.maximum(0, mass_true[f_mask] - mass_pred[f_mask]).mean()
                - np.maximum(0, mass_true[m_mask] - mass_pred[m_mask]).mean()))
NRD = float(abs(np.minimum(0, mass_true[f_mask] - mass_pred[f_mask]).mean()
                - np.minimum(0, mass_true[m_mask] - mass_pred[m_mask]).mean()))
print(f"\nSP (F-M mean pred diff)      = {SP:+.3f} kg   (0 = ideal)")
print(f"PRD (underestimation parity) = {PRD:.3f} kg   (0 = ideal)")
print(f"NRD (overestimation parity)  = {NRD:.3f} kg   (0 = ideal)")

torch.save({"model_state": model.state_dict(), "ch_mean": ch_mean, "ch_std": ch_std,
            "y_mean": y_mean, "y_std": y_std,
            "latent_agn": LATENT_AGN, "latent_sex": LATENT_SEX,
            "seq_len": X.shape[1], "n_channels": X.shape[-1],
            "channel_names": list(map(str, channel_names))}, OUT_DIR / "dvae_emg.pt")
np.savez(OUT_DIR / "test_outputs.npz",
         mu_agn=mu_a_test, mu_sex=mu_s_test,
         mass_true=mass_true, mass_pred=mass_pred,
         y_true_cls=y_true_cls, y_pred_cls=y_pred_cls,
         sex_main_pred=sex_main_pred, sex_adv_pred=sex_adv_pred,
         participant=participant[test_mask], sex=test_sex)
with open(OUT_DIR / "summary.json", "w") as f:
    json.dump({"mae_kg": float(mae), "rmse_kg": float(rmse),
               "nearest_acc": float(acc), "nearest_bal_acc": float(bal_acc),
               "sex_main_acc": float(sex_main_acc), "sex_from_agn_acc": float(sex_adv_acc),
               "SP": SP, "PRD": PRD, "NRD": NRD,
               "confusion_matrix": confusion_matrix(y_true_cls, y_pred_cls).tolist(),
               "fairness": fair_rows,
               "config": {"latent_agn": LATENT_AGN, "latent_sex": LATENT_SEX,
                          "beta": BETA, "lambda_main": LAMBDA_MAIN,
                          "lambda_adv": LAMBDA_ADV, "grl_lambda": GRL_LAMBDA,
                          "batch_size": BATCH_SIZE, "epochs": EPOCHS, "lr": LR, "seed": SEED}}, f, indent=2)
fair_df.to_csv(OUT_DIR / "fairness.csv", index=False)
print("Saved to:", OUT_DIR)

   sex   n   mae_kg  rmse_kg  mean_signed_err_kg  over_rate  under_rate  nearest_acc
Female 479 2.342393 2.897431            0.206072   0.538622    0.461378     0.294363
  Male 480 2.399404 3.069184           -0.222789   0.466667    0.533333     0.322917

SP (F-M mean pred diff)      = +0.429 kg   (0 = ideal)
PRD (underestimation parity) = 0.243 kg   (0 = ideal)
NRD (overestimation parity)  = 0.186 kg   (0 = ideal)
Saved to: C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\VAE\Results\dvae_emg
